In [1]:
import pandas as pd
import sys
import numpy as np
import warnings

sys.path.append("/Users/ejowik001/Desktop/Github/Nowcasting/kedro/refinery/dependencies/")

In [2]:
# from utils import cast_spec_to_dict
from load_spec import load_spec
from load_data import load_data


In [3]:
pd.set_option("display.max_columns", None)
np.set_printoptions(threshold=sys.maxsize)

In [4]:
ds = pd.read_excel("../data/02_intermediate/harmonized_time_series.xlsx", header=None)

In [5]:
# ds_spec = pd.read_csv("../data/02_intermediate/variable.csv")
# Spec = cast_spec_to_dict(ds_spec)
Spec = load_spec("../data/0_source/Spec_US_example.xls")

Table 1: Model specification
              SeriesID                   SeriesName                 Units  \
0               PAYEMS           Payroll Employment  Thousands of Persons   
1               JTSJOL                 Job Openings             Thousands   
2             CPIAUCSL         Consumer Price Index                 Index   
3              DGORDER         Durable Goods Orders           $, Millions   
4                RSAFS                 Retail Sales           $, Millions   
5               UNRATE            Unemployment Rate                     %   
6                HOUST               Housing Starts    Thousands of Units   
7               INDPRO        Industrial Production                 Index   
8              DSPIC96              Personal Income   Chained $, Billions   
9              BOPTEXP                      Exports           $, Millions   
10             BOPTIMP                      Imports           $, Millions   
11             TTLCONS        Construction Spen

/Users/ejowik001/Desktop/Github/Nowcasting/kedro/refinery/dependencies/load_spec.py:41: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  blocks = raw_data[block_cols].fillna(0).astype(int).values


In [6]:
import numpy as np
import warnings


def load_data(ds, Spec, sample=None, load_excel=False):
    """
    Load vintage of data from file and format as structure

    Parameters:
        datafile (str): Filename of Microsoft Excel workbook file
        Spec (dict): Model specification containing SeriesID and other info
        sample (float, optional): Sample period start date in numeric form
        load_excel (bool, optional): Flag to force loading from Excel

    Returns:
        X (np.ndarray): T x N numeric array, transformed dataset
        Time (np.ndarray): T x 1 numeric array, date number with observation dates
        Z (np.ndarray): T x N numeric array, raw (untransformed) dataset
    """
    print('Loading data...')

    Z, Time, Mnem = read_data(ds)

    # Sort data based on model specification
    Z = sort_data(Z, Mnem, Spec)
    
    # Transform data based on model specification
    X, Time, Z, header = transform_data(Z, Time, Spec)

    # Drop data not in estimation sample
    if sample is not None:
        X, Time, Z = drop_data(X, Time, Z, sample)

    # Z = np.vstack([header, Z])
    # X = np.vstack([header, X])

    return X, Time, Z, header


def read_data(ds):
    """
    Read data from Microsoft Excel workbook file

    Parameters:
        datafile (str): Filename of the Excel file

    Returns:
        Z (np.ndarray): Raw (untransformed) observed data
        Time (np.ndarray): Observation periods for the time series data
        Mnem (list): Series ID for each variable
    """
    # df = pd.read_excel(datafile, sheet_name='data', header=None, engine="openpyxl")
    Mnem = ds.iloc[0, 1:].tolist()

    # if os.name == 'nt':  # Check if the operating system is Windows
    #     Time = pd.to_datetime(df.iloc[1:, 0], format='%m/%d/%Y').astype(np.int64) // 10**9
    #     Z = df.iloc[1:, 1:].to_numpy()
    # else:
    #     Time = (df.iloc[1:, 0] + pd.Timestamp('1899-12-31').to_julian_date()).to_numpy()
    #     Z = df.iloc[:, 1:].to_numpy()
    Time = ds.iloc[1:, 0].to_numpy()
    Z = ds.iloc[:, 1:].to_numpy()
    return Z, Time, Mnem


def sort_data(Z, Mnem, Spec):
    """
    Sort series by order of model specification

    Parameters:
        Z (np.ndarray): Raw data
        Mnem (list): Series ID for each variable
        Spec (dict): Model specification

    Returns:
        Z (np.ndarray): Sorted data according to Spec.SeriesID
    """
    in_spec = np.isin(Mnem, Spec['seriesid'])
    Mnem = [mnem for mnem, keep in zip(Mnem, in_spec) if keep]
    Z = Z[:, in_spec]

    # Sort series by ordering of Spec
    N = len(Spec['seriesid'])
    permutation = [Mnem.index(spec_id) for spec_id in Spec['seriesid']]

    Mnem = [Mnem[i] for i in permutation]
    Z = Z[:, permutation]

    return Z


# def transform_data(Z, Time, Spec):
#     """
#     Transforms each data series based on Spec.Transformation

#     Parameters:
#         Z (np.ndarray): Raw (untransformed) observed data
#         Time (np.ndarray): Observation periods for the time series data
#         Spec (dict): Model specification

#     Returns:
#         X (np.ndarray): Transformed data (stationary to enter DFM)
#         Time (np.ndarray): Adjusted time data
#         Z (np.ndarray): Adjusted raw data
#     """
#     T, N = Z.shape

#     X = np.full((T, N), np.nan)
#     for i in range(N):
#         formula = Spec["transformation"][i]
#         freq = Spec["frequency"][i]
#         step = 1 if freq == "m" else 3
#         t1 = step
#         n = step / 12
#         series = Spec["seriesname"][i]

#         if formula == "lin":  # Levels (No Transformation)
#             X[:, i] = Z[:, i]
#         elif formula == "chg":  # Change (Difference)
#             X[t1:, i] = np.concatenate(([np.nan], Z[t1:, i] - Z[:-t1, i]))
#         elif formula == "ch1":  # Year over Year Change (Difference)
#             if T > 12:
#                 X[12 + t1 :, i] = Z[12 + t1 :, i] - Z[:-12, i]
#         elif formula == "pch":  # Percent Change
#             X[t1:, i] = 100 * np.concatenate(([np.nan], Z[t1:, i] / Z[:-t1, i] - 1))
#         elif formula == "pc1":  # Year over Year Percent Change
#             if T > 12:
#                 X[12 + t1 :, i] = 100 * (Z[12 + t1 :, i] / Z[:-12, i] - 1)
#         elif formula == "pca":  # Percent Change (Annual Rate)
#             X[t1:, i] = 100 * np.concatenate(
#                 ([np.nan], (Z[t1:, i] / Z[:-step, i]) ** (1 / n) - 1)
#             )
#         elif formula == "log":  # Natural Log
#             X[:, i] = np.log(Z[:, i])
#         else:
#             warnings.warn(
#                 f"Transformation '{formula}' not found for {series}. Using untransformed data."
#             )
#             X[:, i] = Z[:, i]

#     # Drop first quarter of observations since transformations cause missing values
#     Time = Time[3:]
#     Z = Z[3:, :]
#     X = X[3:, :]

#     return X, Time, Z


def transform_data(Z, Time, Spec):
    """
    Transforms each data series based on Spec.Transformation

    Parameters:
        Z (np.ndarray): Raw (untransformed) observed data
        Time (np.ndarray): Observation periods for the time series data
        Spec (dict): Model specification

    Returns:
        X (np.ndarray): Transformed data (stationary to enter DFM)
        Time (np.ndarray): Adjusted time data
        Z (np.ndarray): Adjusted raw data
    """
    header = Z[0, :]
    Z = np.float64(Z[1:, :])

    T, N = Z.shape

    X = np.full((T, N), np.nan)

    for i in range(N):
        formula = Spec["transformation"][i]
        freq = Spec["frequency"][i]
        step = 1 if freq == "m" else 3
        t1 = step
        n = step / 12

        assert header[i]== Spec["seriesid"][i]
        series = Spec["seriesname"][i]

        # first_valid_index = np.argwhere(np.isfinite(Z[:, i])).ravel()[0]
        
         # Apply transformations based on formula
        if formula == 'lin':  # Levels (No Transformation)
            X[:, i] = Z[:, i]
        elif formula == 'chg':  # Change (Difference)
            # X[(t1-1):T, i] = np.concatenate(([np.nan], Z[(t1-1+step):T, i] - Z[(t1-1):(T-t1), i]))
            X[(t1-1+step):T, i] = (Z[(t1-1+step):T, i] - Z[(t1-1):(T-t1), i])
        elif formula == 'ch1':  # Year over Year Change (Difference)
            if T > 12:
                # print(Z[(12+first_valid_index):T, i] - Z[first_valid_index:(T - 12), i])
                X[(12+t1-1):T, i] = Z[(12+t1-1):T, i] - Z[(t1-1):(T - 12), i]
        elif formula == 'pch':  # Percent Change
            # X[(t1-1):T, i] = 100 * np.concatenate(
            #     ([np.nan], Z[(t1-1+step):T, i] / Z[(t1-1):(T-t1), i] - 1)
            # )
            X[(t1-1+step):T, i] = 100 * (Z[(t1-1+step):T, i] / Z[(t1-1):(T-t1), i] - 1)
        elif formula == 'pc1':  # Year over Year Percent Change
            if T > 12:
                # Year over Year Percent Change, handle division by zero
                X[(12+t1-1):T, i] = 100 * (Z[(12+t1-1):T, i] / Z[(t1-1):(T-12), i] - 1)
        elif formula == 'pca':  # Percent Change (Annual Rate)
            # X[(t1-1):T, i] = 100 * np.concatenate(
            #     ([np.nan], (Z[(t1-1+step):T, i] / Z[(t1-1):(T-step), i]) ** (1 / n) - 1)
            # )
            X[(t1-1+step):T, i] = 100 * ((Z[(t1-1+step):T, i] / Z[(t1-1):(T-step), i]) ** (1 / n) - 1)
        elif formula == 'log':  # Natural Log
            X[:, i] = np.log(Z[:, i])
        else:
            warnings.warn(f"Transformation '{formula}' not found for {series}. Using untransformed data.")
            X[:, i] = Z[:, i]

    # Drop first quarter of observations since transformations cause missing values
    Time = Time[3:]
    Z = Z[3:, :]
    X = X[3:, :]

    return X, Time, Z, header


def drop_data(X, Time, Z, sample):
    """
    Remove data not in estimation sample

    Parameters:
        X (np.ndarray): Transformed data
        Time (np.ndarray): Time data
        Z (np.ndarray): Raw data
        sample (float): Sample period start date in numeric form

    Returns:
        X (np.ndarray): Filtered transformed data
        Time (np.ndarray): Filtered time data
        Z (np.ndarray): Filtered raw data
    """
    idx_drop = Time < sample

    Time = Time[~idx_drop]
    X = X[~idx_drop, :]
    Z = Z[~idx_drop, :]

    return X, Time, Z


In [7]:
sample_start = "2000-01-01"
sample_start = pd.to_datetime(sample_start, format="%Y-%m-%d")

test = ds[np.where(ds.loc[0].isin(["ReferenceDate"]+list(Spec["seriesid"])))[0]]
X, Time, Z, header = load_data(test, Spec)
df = pd.DataFrame(X, columns=header, index=Time)
df

Loading data...


,PAYEMS,JTSJOL,CPIAUCSL,DGORDER,RSAFS,UNRATE,HOUST,INDPRO,DSPIC96,BOPTEXP,BOPTIMP,TTLCONS,IR,CPILFESL,PCEPILFE,PCEPI,PERMIT,TCU,BUSINV,IQ,GACDISA066MSFRBNY,PCEC96,GACDFSA066MSFRBPHI,GDPC1,ULCNFB
1855-03-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1855-04-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1855-05-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1855-06-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1855-07-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-04-01,108.0,-458.0,0.312910,0.797170,-0.150198,0.1,6.004619,-0.068963,0.159196,-1.854620,-1.740932,0.094716,0.854093,0.291761,0.334092,0.339144,-45.0,-0.1192,-0.128033,0.672043,-14.3,-0.025495,15.5,2.988846,0.423589
2024-05-01,216.0,-436.0,0.005747,0.227228,0.231684,0.1,-4.502542,0.749357,-0.059678,0.626436,2.605788,1.283284,-0.141143,0.163087,0.254873,0.261439,-41.0,0.5152,0.344159,-0.734312,-15.6,0.532973,4.5,NaN,NaN
2024-06-01,118.0,311.0,-0.056190,0.088421,-0.278145,0.1,1.064639,0.062200,0.325762,-0.580983,-0.332427,0.232621,0.000000,0.064751,0.101690,0.007311,55.0,-0.0222,0.495008,-0.201748,-6.0,0.263171,1.3,NaN,NaN
2024-07-01,89.0,-320.0,0.154928,-6.871694,1.147434,0.2,-6.922498,-0.943069,0.086038,1.716323,0.650409,0.035928,0.212014,0.165229,0.163031,0.060920,-48.0,-0.8126,0.277588,0.539084,-6.6,0.376960,13.9,NaN,NaN


In [8]:
sample_start = "2000-01-01"
sample_start = pd.to_datetime(sample_start, format="%Y-%m-%d")

X, Time, Z, header = load_data(test, Spec, sample_start)

# # summarize data
# summarize(X.astype(float), Time, Spec)

# # Prepare data -----------------------------------------------------------
# T, N = X.shape  # Gives dimensions for data input
# indNaN = np.isnan(X)  # Returns location of NaNs
# rem = np.sum(indNaN, axis=0) > T * 0.8  # Returns columns sum for NaN values. Marks true for rows with more than 80% NaN
# X = X[:, ~rem]

# # Mx = np.nanmean(X, axis=0)
# # Wx = np.nanstd(X, axis=0)
# # xNaN = (X - Mx) / Wx  # Standardize series

# optNaN = {"method": 2, "k": 3}
# x_est, indNaN, nanLE = remNaNs_spline(X, optNaN)  # Impute series
# X[np.isnan(X)] = x_est[np.isnan(X)]

# Spec = cast_spec_to_dict(ds_spec.loc[ds_spec["SeriesID"].isin(list(compress(header, ~rem)))])


Loading data...


In [9]:
import numpy as np
import warnings
import numpy as np
import warnings
from datetime import datetime


def retransform_data(
    X: np.ndarray,
    Z: np.ndarray,
    Time: np.ndarray,
    Spec: dict,
    header: list,
    cutoff_date: datetime,
) -> np.ndarray:
    """
    Retransforms stationary data series back to their original form based on specified transformations.

    This function takes stationary (transformed) data and reverts it to its original scale using the
    transformation specifications provided. It handles various types of transformations such as linear
    levels, changes, percent changes, and logarithmic transformations.

    Parameters
    ----------
    X : np.ndarray
        Transformed data (stationary), with shape (H, N), where H is the number of periods and N is the
        number of series.
    Z : np.ndarray
        Nominal (raw) data in base units, with shape (H, N).
    Time : np.ndarray
        Observation periods for the time series data, typically as datetime objects.
    Spec : dict
        Model specification containing transformation details. It should include the following keys:
            - "transformation": List of transformation types for each series.
            - "frequency": List of frequencies (e.g., 'm' for monthly) for each series.
            - "seriesid": List of series identifiers.
            - "seriesname": List of series names.
    header : list
        Original data headers corresponding to each series.
    cutoff_date : datetime
        The date marking the beginning of the transformed records. All data from this date onward
        are treated as forecasts.

    Returns
    -------
    np.ndarray
        Retransformed data in original form, with shape (H, N). This array includes both historical
        data from `Z` and retransformed forecasts from `X`.

    Raises
    ------
    AssertionError
        If any header in `header` does not match the corresponding `Spec["seriesid"]`.
    Warning
        If an unknown transformation type is encountered for any series, the function will issue a
        warning and use the untransformed data for that series.

    Examples
    --------
    >>> import numpy as np
    >>> from datetime import datetime
    >>> X = np.random.randn(100, 3)
    >>> Z = np.random.randn(120, 3)
    >>> Time = np.array([datetime(2020, 1, 1) + np.timedelta64(i, 'M') for i in range(120)])
    >>> Spec = {
    ...     "transformation": ["lin", "chg", "log"],
    ...     "frequency": ["m", "m", "m"],
    ...     "seriesid": ["series1", "series2", "series3"],
    ...     "seriesname": ["Series 1", "Series 2", "Series 3"]
    ... }
    >>> header = ["series1", "series2", "series3"]
    >>> cutoff_date = datetime(2023, 1, 1)
    >>> V_final = retransform_data(X, Z, Time, Spec, header, cutoff_date)
    """

    # Filter indexes that denote predictions
    c_idx = np.where(Time >= cutoff_date)[0]

    # Get the number of periods of predictions
    T = c_idx.shape[0]
    # Get number of periods, series
    H = X.shape[0]
    N = X.shape[1]
    # Initialize T x N matrix filled with NaN's
    V = np.full((T, N), np.nan)

    for i in range(N):
        formula = Spec["transformation"][i]
        freq = Spec["frequency"][i]
        step = 1 if freq == "m" else 3
        t1 = step
        n = step / 12
        assert header[i] == Spec["seriesid"][i]
        series = Spec["seriesname"][i]

        # Apply inverse transformations based on formula
        if formula == "lin":  # Levels (No Transformation)
            V[:, i] = X[:, i][c_idx]
        elif formula == "chg":  # Change (Difference)
            V[0:T:step, i] = np.cumsum(X[c_idx[0]:H:step, i]) + Z[c_idx[0] - step, i]  # Assuming X[t1-step] is a last historical value
        elif formula == "ch1":
            V[0:T:step, i] = np.add(Z[c_idx[0]-12:H-12:step, i], X[c_idx[0]:H:step, i])
        elif formula == "pch":  # Percent Change
            V[0:T:step, i] = np.cumprod(1 + (X[c_idx[0]:H:step, i] / 100)) * Z[c_idx[0] - step, i]  # Assuming X[t1-step] is a last historical value
        elif formula == "pc1":  # Year over Year Percent Change
            V[0:T:step, i] = np.multiply(
                1 + (X[c_idx[0]:H:step, i] / 100), Z[c_idx[0]-12:H-12:step, i]
            )
        elif formula == "pca":  # Percent Change (Annual Rate)
            V[0:T:step, i] = np.cumprod((1 + X[c_idx[0]:H:step, i] / 100) ** n) * Z[c_idx[0] - step, i]
        elif formula == "log":  # Natural Log
            V[:, i] = np.exp(X[:, i][c_idx])
        else:
            warnings.warn(
                f"Transformation '{formula}' not found for {series}. Using untransformed data."
            )
            V[:, i] = X[:, i][c_idx]

        V_final = np.full((H, N), np.nan)
        for _ in range(N):
            V_final[:, _] = np.concatenate((Z[: c_idx[0], _], V[:, _]))
    return V_final

# Example usage retransform_data(X, Z, Time, Spec, header, datetime(2023, 1, 1))


In [12]:
from datetime import datetime

R = retransform_data(X, Z, Time, Spec, header, datetime(2023, 1, 1))
R_df = pd.DataFrame(R, columns=header, index=Time)

Z_df = pd.DataFrame(Z, columns=header, index=Time)

In [13]:
Z_df

,PAYEMS,JTSJOL,CPIAUCSL,DGORDER,RSAFS,UNRATE,HOUST,INDPRO,DSPIC96,BOPTEXP,BOPTIMP,TTLCONS,IR,CPILFESL,PCEPILFE,PCEPI,PERMIT,TCU,BUSINV,IQ,GACDISA066MSFRBNY,PCEC96,GACDFSA066MSFRBPHI,GDPC1,ULCNFB
2000-01-01,131009.0,NaN,169.300,196344.0,268044.0,4.0,1636.0,91.4092,9735.9,86192.0,112131.0,789431.0,97.8,179.300,74.128,72.763,1727.0,82.0932,1137260.0,99.2,NaN,7988.5,11.3,13878.147,85.710
2000-02-01,131120.0,NaN,170.000,201360.0,272020.0,4.1,1737.0,91.7245,9799.9,86001.0,113133.0,784940.0,99.7,179.400,74.306,72.961,1692.0,82.0943,1140023.0,99.6,NaN,8065.3,14.5,NaN,NaN
2000-03-01,131604.0,NaN,171.000,183911.0,275192.0,4.0,1604.0,92.0830,9837.9,86540.0,116334.0,793737.0,99.9,180.000,74.415,73.191,1651.0,82.1340,1146435.0,100.0,NaN,8110.8,19.4,NaN,NaN
2000-04-01,131883.0,NaN,170.900,192130.0,271046.0,3.8,1626.0,92.6659,9864.0,88115.0,118672.0,809459.0,98.5,180.300,74.568,73.505,1597.0,82.3741,1150666.0,100.0,NaN,8101.6,10.0,14130.908,84.232
2000-05-01,132106.0,NaN,171.200,195044.0,271394.0,4.0,1575.0,92.9347,9913.7,89359.0,117957.0,804766.0,98.8,180.700,74.617,73.444,1543.0,82.3350,1157552.0,100.2,NaN,8139.0,13.9,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-04-01,158214.0,8355.0,313.207,282096.0,702681.0,3.9,1377.0,102.4479,16924.3,260681.0,328653.0,2135771.0,141.7,317.622,121.629,122.782,1440.0,77.7263,2537490.0,149.8,-14.3,15685.6,15.5,23223.906,120.245
2024-05-01,158430.0,7919.0,313.225,282737.0,704309.0,4.0,1315.0,103.2156,16914.2,262314.0,337217.0,2163179.0,141.5,318.140,121.939,123.103,1399.0,78.2415,2546223.0,148.7,-15.6,15769.2,4.5,NaN,NaN
2024-06-01,158548.0,8230.0,313.049,282987.0,702350.0,4.1,1329.0,103.2798,16969.3,260790.0,336096.0,2168211.0,141.5,318.346,122.063,123.112,1454.0,78.2193,2558827.0,148.4,-6.0,15810.7,1.3,NaN,NaN
2024-07-01,158637.0,7910.0,313.534,263541.0,710409.0,4.3,1237.0,102.3058,16983.9,265266.0,338282.0,2168990.0,141.8,318.872,122.262,123.187,1406.0,77.4067,2565930.0,149.2,-6.6,15870.3,13.9,NaN,NaN


In [14]:
R_df

,PAYEMS,JTSJOL,CPIAUCSL,DGORDER,RSAFS,UNRATE,HOUST,INDPRO,DSPIC96,BOPTEXP,BOPTIMP,TTLCONS,IR,CPILFESL,PCEPILFE,PCEPI,PERMIT,TCU,BUSINV,IQ,GACDISA066MSFRBNY,PCEC96,GACDFSA066MSFRBPHI,GDPC1,ULCNFB
2000-01-01,131009.0,NaN,169.300,196344.0,268044.0,4.0,1636.0,91.4092,9735.9,86192.0,112131.0,789431.0,97.8,179.300,74.128,72.763,1727.0,82.0932,1137260.0,99.2,NaN,7988.5,11.3,13878.147,85.710
2000-02-01,131120.0,NaN,170.000,201360.0,272020.0,4.1,1737.0,91.7245,9799.9,86001.0,113133.0,784940.0,99.7,179.400,74.306,72.961,1692.0,82.0943,1140023.0,99.6,NaN,8065.3,14.5,NaN,NaN
2000-03-01,131604.0,NaN,171.000,183911.0,275192.0,4.0,1604.0,92.0830,9837.9,86540.0,116334.0,793737.0,99.9,180.000,74.415,73.191,1651.0,82.1340,1146435.0,100.0,NaN,8110.8,19.4,NaN,NaN
2000-04-01,131883.0,NaN,170.900,192130.0,271046.0,3.8,1626.0,92.6659,9864.0,88115.0,118672.0,809459.0,98.5,180.300,74.568,73.505,1597.0,82.3741,1150666.0,100.0,NaN,8101.6,10.0,14130.908,84.232
2000-05-01,132106.0,NaN,171.200,195044.0,271394.0,4.0,1575.0,92.9347,9913.7,89359.0,117957.0,804766.0,98.8,180.700,74.617,73.444,1543.0,82.3350,1157552.0,100.2,NaN,8139.0,13.9,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-04-01,158214.0,8355.0,313.207,282096.0,702681.0,3.9,1377.0,102.4479,16924.3,260681.0,328653.0,2135771.0,141.7,317.622,121.629,122.782,1440.0,77.7263,2537490.0,149.8,-14.3,15685.6,15.5,23223.906,120.245
2024-05-01,158430.0,7919.0,313.225,282737.0,704309.0,4.0,1315.0,103.2156,16914.2,262314.0,337217.0,2163179.0,141.5,318.140,121.939,123.103,1399.0,78.2415,2546223.0,148.7,-15.6,15769.2,4.5,NaN,NaN
2024-06-01,158548.0,8230.0,313.049,282987.0,702350.0,4.1,1329.0,103.2798,16969.3,260790.0,336096.0,2168211.0,141.5,318.346,122.063,123.112,1454.0,78.2193,2558827.0,148.4,-6.0,15810.7,1.3,NaN,NaN
2024-07-01,158637.0,7910.0,313.534,263541.0,710409.0,4.3,1237.0,102.3058,16983.9,265266.0,338282.0,2168990.0,141.8,318.872,122.262,123.187,1406.0,77.4067,2565930.0,149.2,-6.6,15870.3,13.9,NaN,NaN
